In [1]:
# Import necessary libraries
!pip install transformers gradio timm inflect phonemizer
!pip install py-espeak-ng

from transformers import pipeline
from PIL import Image
import requests
from io import BytesIO
import os

# Define helper functions
def load_image_from_url(url):
    response = requests.get(url)
    img = Image.open(BytesIO(response.content))
    return img

def render_results_in_image(image, results):
    draw = ImageDraw.Draw(image)
    for result in results:
        box = result['box']
        label = result['label']
        score = result['score']
        draw.rectangle([(box['xmin'], box['ymin']), (box['xmax'], box['ymax'])], outline="red", width=3)
        draw.text((box['xmin'], box['ymin']), f"{label} ({score:.2f})", fill="red")
    return image

# Load Image
img_url = "https://path-to-your-image.jpg"
raw_image = load_image_from_url(img_url)
raw_image

# Build the object-detection pipeline using 🤗 Transformers Library
od_pipe = pipeline(task="object-detection", model="facebook/detr-resnet-50")

# Resize image
raw_image = raw_image.resize((600, 400))

# Detect objects in the image
pipeline_output = od_pipe(raw_image)

# Render results on the image
propossed_image = render_results_in_image(raw_image.copy(), pipeline_output)
propossed_image

# Save cropped objects
def crop_and_save_objects(image, pipeline_output, save_dir="cropped_objects"):
    if not os.path.exists(save_dir):
        os.makedirs(save_dir)

    for i, result in enumerate(pipeline_output):
        box = result['box']
        label = result['label']
        cropped_image = image.crop((box['xmin'], box['ymin'], box['xmax'], box['ymax']))
        cropped_image.save(os.path.join(save_dir, f"{label}_{i}.png"))

crop_and_save_objects(raw_image, pipeline_output)

  Using cached aiofiles-23.2.1-py3-none-any.whl.metadata (9.7 kB)
  Using cached orjson-3.10.15-cp311-cp311-macosx_10_15_x86_64.macosx_11_0_arm64.macosx_10_15_universal2.whl.metadata (41 kB)
  Using cached pandas-2.2.3-cp311-cp311-macosx_11_0_arm64.whl.metadata (89 kB)
  Using cached pydub-0.25.1-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached semantic_version-2.10.0-py2.py3-none-any.whl.metadata (9.7 kB)
  Using cached tomlkit-0.13.2-py3-none-any.whl.metadata (2.7 kB)
  Using cached pytz-2025.1-py2.py3-none-any.whl.metadata (22 kB)
  Using cached shellingham-1.5.4-py2.py3-none-any.whl.metadata (3.5 kB)
  Using cached rich-13.9.4-py3-none-any.whl.metadata (18 kB)
  Using cached uritemplate-4.1.1-py2.py3-none-any.whl.metadata (2.9 kB)
  Using cached colorama-0.4.6-py2.py3-none-any.whl.metadata (17 kB)
  Using cached markdown_it_py-3.0.0-py3-none-any.whl.metadata (6.9 kB)
  Using cached mdurl-0.1.2-py3-none-any.whl.metadata (1.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.2/

/Users/phamthanh/Desktop/docker/model_ai/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ConnectionError: HTTPSConnectionPool(host='path-to-your-image.jpg', port=443): Max retries exceeded with url: / (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x1066634d0>: Failed to resolve 'path-to-your-image.jpg' ([Errno 8] nodename nor servname provided, or not known)"))